In [1]:
# 关注一个序列
# 不是每个观察值都是同样重要 (观察不同的变化)
# 想只记住相关的观察需要
#  - 能关注的机制(更新门), 这个数据比较重要, 需要这个数据去更新
#  - 能遗忘的机制(重置门), 这个数据不重要, 把输入/隐藏状态丢掉一点

In [ ]:
# 门
# 有一个输入x_t, 一个隐藏的状态 H_t-1, 
# 两个门: R_t(重置门, Reset Gate); Z_t(更新门, Update Gate)
# R_t = σ(X_t*W_xr + H_t-1*W_hr + b_r) 和之前的隐藏层基本一样
# Z_t = σ(X_t*W_xz + H_t-1*W_hz + b_z)
# σ是sigmod作为激活函数
# 流程: x_t 和 H_t-1进来, concat, 分别放到R_t和Z_t里

In [2]:
# 候选隐藏状态
# 另外一个是候选隐藏状态
# ℏ_t = tanh(X_t*W_xh + (R_t⊙H_t-1)W_hh + b_h)
# 这里的R_t的大小和H_t-1的大小一样, 因为这里R_t的作用去筛选哪些H_t-1的信息是不需要的
# 也就是说R_t是一个大小和H_t-1一样的矩阵, 有[0,1]之间的元素, 如果不需要哪个信息, 则R_t对应那个信息为0, 相对应, 越需要越接近1
# 这里R_t⊙H_t-1也就是筛选了哪些H_t-1需要遗忘的信息, 将需要的H_t-1的信息保留, 传到候选隐藏状态

In [3]:
# 隐状态
# H_t = Z_t⊙H_t-1 + (1-Z_t)⊙ℏ_t
# 这里的Z_t就是更新门, 同样, 这里的Z_t的作用去筛选哪些H_t-1的信息是需要的
# Z_t的大小和H_t-1一样, 在[0,1]之间, 如果需要则为1, 不需要则为0
# 所以Z_t⊙H_t-1就是得到哪些需要保留的旧状态
# (1-Z_t)计算出每个维度上新信息的占比, ⊙ℏ_t则是根据这些新信息的占比, 引入新的候选隐藏状态

In [4]:
# 总结
# GRU引入了两个额外的门, 每个门可学习的参数和RNN一样多
# 这两个门都是控制单元, 输出0~1之间的值
# Reset gate控制旧信息和新信息的占比, 并将新的候选信息计算出
# Update gate控制新信息的占比
# 最后用Update gate计算出需要的旧信息的占比, 并相应引入新的候选隐藏层

In [8]:
import torch
from torch import nn
from d2l import torch as d2l

batch_size, num_steps = 32, 35 # num_steps 指可以预测多少个
train_iter, vocab = d2l.load_data_time_machine(32, 35)

1.2.4


In [2]:
# 初始化模型参数
def get_params(vocab_size, num_hiddens, device):
    num_inputs = num_hiddens = vocab_size
    
    def normal(shape):
        return torch.randn(size=shape, device=device) * 0.01
    
    def three():
        return (normal((num_inputs, num_hiddens)),
                normal((num_hiddens, num_hiddens)),
                torch.zeros(num_hiddens, device=device))
    
    W_xz, W_hz, b_z = three()
    W_xr, W_hr, b_r = three()
    W_xh, W_hh, b_h = three()
    W_hq = normal((num_hiddens, num_outputs)) # 输出多类分类
    b_q = torch.zeros(num_outputs, device=device)
    params = [W_xz, W_hz, b_z, W_xr, W_hr, b_r, W_xh, W_hh, b_h, W_hq, b_q]
    for param in params:
        param.require_grad_(True)
    return params

In [4]:
# 定义隐藏状态的初始化函数
def init_gru_state(batch_size, num_hiddens, device):
    return (torch.zeros((batch_size, num_hiddens), device=device))

In [5]:
# 定义门控循环单元模型
def gru(inputs, state, params):
    W_xz, W_hz, b_z, W_xr, W_hr, b_r, W_xh, W_hh, b_h, W_hq, b_q = params
    H, = state
    outputs = []
    for X in inputs:
        Z = torch.sigmoid((X @ W_xz) + (H @ W_hz) + b_z) # 这里的@是矩阵乘法
        R = torch.sigmoid((X @ W_xr) + (H @ W_hr) + b_r)
        H_tilda = torch.tanh((X @ W_hr) + ((R * H) @ W_hh) + b_h) 
        # 到这一时刻前, H都是我们公式中的H_t-1, 直到下面更新H才是新的H
        H = Z * H + (1 - Z) * H_tilda
        Y = H @ W_bq + b_q
        outputs.append(Y)
    return torch.cat(outputs, dim=0), (H,)

In [6]:
# 训练
vocab_size, num_hiddens, device = len(vocab), 256, d2l.try_gpu()
num_epoch, lr = 500, 1
model = d2l.RNNModelScratch(len(vocab), num_hiddens, device, get_params,
                            init_gru_state, gru)
d2l.train_ch8(model, train_iter, vocab, lr, num_epochs, device)


KeyboardInterrupt



In [ ]:
# 简介实现
num_inputs = vocab_size
gru_layer = nn.GRU(num_inputs, num_hiddens)
model = d2l.RNNModel(gru_layer, len(vocab))
model.to(device)
d2l.train_ch8(model, train_iter, vocab, lr, num_epochs, device)